In [13]:
import os
import sys
import cv2
import numpy as np

sys.path.append(os.path.dirname(os.getcwd()))
sys.path.insert(
    0,
    os.path.abspath(
        os.path.join(os.path.dirname(os.getcwd()), os.pardir)
    )
)

from src.models.discriminator import CNNClassifier
from src.dataloader.dataloaders import MadisonDatasetLabeled
from src.models.unet import BaseUNet
from src.utils.viz_utils import visualize_predictions
from src.utils.args_utils import train_arg_parser
from src.evaluation.segmentation_metrics import dice_coefficient
from src.utils.variable_utils import PLOT_DIRECTORY, TRAINING_LOO, VALIDATION_LOO

In [21]:
train_dir = '/home/miguel/GI/1 - Segmentation/UNet-and-Synthesis-Results/train-folders/train-200'
train_dataset = MadisonDatasetLabeled(train_dir, augment=True)

Number of images: 6299
Number of masks: 6299
Number of fake images: 0


In [22]:
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedGroupKFold

In [23]:
skf = StratifiedGroupKFold(n_splits=5, shuffle=True)

In [24]:
from torch.utils.data import DataLoader, Subset

In [25]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [26]:
for fold, (train_idx, val_idx) in enumerate(kf.split(range(len(train_dataset))), 1):
    print(f"Fold {fold}: train={len(train_idx)}  val={len(val_idx)}")

    train_subset = Subset(train_dataset, train_idx)
    val_subset   = Subset(train_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_subset,   batch_size=16, shuffle=False, num_workers=4)

Fold 1: train=5039  val=1260
Fold 2: train=5039  val=1260
Fold 3: train=5039  val=1260
Fold 4: train=5039  val=1260
Fold 5: train=5040  val=1259


In [30]:
sample_path = next(iter(train_subset))[-1]

In [34]:
sample_path.split('/')[-1].split('_slice')[0]

'case101_day20'

In [13]:
labels = []
groups = []
for img_path, mask_path in zip(train_dataset.image_paths, train_dataset.mask_paths):
    # --- derive your label; here: any nonzero mask pixels?
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
    label = int(mask.sum() > 0)             # 1 if there's any foreground, else 0
    labels.append(label)
    
    # --- derive your group id; e.g. split on "_" and take first token
    #     adjust to whatever grouping makes sense for you
    group_id = os.path.basename(img_path).split('_')[0]
    groups.append(group_id)

labels = np.array(labels)
groups = np.array(groups)

# 3) StratifiedGroupKFold
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X=np.zeros(len(train_dataset)),
                                                    y=labels,
                                                    groups=groups), 1):
    print(f"Fold {fold}:")
    print(f"  Train samples: {len(train_idx)},  Val samples: {len(val_idx)}")

Fold 1:
  Train samples: 1391,  Val samples: 362
Fold 2:
  Train samples: 1413,  Val samples: 340
Fold 3:
  Train samples: 1234,  Val samples: 519
Fold 4:
  Train samples: 1389,  Val samples: 364
Fold 5:
  Train samples: 1585,  Val samples: 168


In [1]:
train_dataset

NameError: name 'train_dataset' is not defined